## Basic Features Extractor

Extract acoustic features from BirdCLEF 2026 training set. Each row includes a `start_sec` timestamp.

**Windowing:** 5s window, 2.5s hop, top 3 loudest by RMS energy.

**Output:** `features_v2_with_start_sec.parquet`

| Feature Group | Dims | Description |
|---|---|---|
| Mel Spectrogram | 3 | dominant band, mean, std |
| RMS Energy | 5 | mean/std/min/max + burst ratio |
| MFCC (20 coeffs) | 80 | mean/std/min/max per coefficient |
| Delta MFCC | 80 | 1st and 2nd order deltas, mean/std |
| Spectral Centroid | 4 | mean/std/min/max |
| Spectral Bandwidth | 2 | mean/std |
| Spectral Rolloff (85% / 95%) | 3 | mean/std |
| Spectral Contrast (7 bands) | 14 | mean/std per band |
| Spectral Flatness | 2 | mean/std |
| Zero Crossing Rate | 2 | mean/std |


- without noise bank, extract features from the original data

In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DATA_ROOT  = '/kaggle/input/competitions/birdclef-2026'
OUTPUT_DIR = '/kaggle/working'

# keep these in sync with trainingv2-version2-crossvalidation
SAMPLE_RATE  = 32_000
SEGMENT_SEC  = 5
K_WINDOWS    = 3
HOP_SEC      = 2.5
MIN_RATING   = 2

DEBUG_MODE   = False
N_DEBUG_ROWS = 10

OUT_PARQUET = os.path.join(OUTPUT_DIR, 'features_v2_with_start_sec.parquet')

In [ ]:
def load_audio_fast(file_path):
    try:
        audio, sr = sf.read(file_path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        # soundfile OGG bug: returns all zeros for some files
        if np.max(np.abs(audio)) == 0:
            audio, sr = librosa.load(file_path, sr=None)
        if sr != SAMPLE_RATE:
            audio = librosa.resample(y=audio, orig_sr=sr, target_sr=SAMPLE_RATE)
        return audio
    except Exception:
        return np.zeros(SAMPLE_RATE * SEGMENT_SEC)


# same as trainingv2 but also returns the exact start times so we can merge later
def top_k_energy_windows(y, sr, segment_sec, k=3, hop_sec=2.5):
    target_len = int(sr * segment_sec)
    if len(y) <= target_len:
        padded = np.pad(y, (0, max(0, target_len - len(y))), 'constant')
        return [padded], [0.0]
    hop    = max(1, int(sr * hop_sec))
    starts = list(range(0, len(y) - target_len + 1, hop))
    windows  = [y[s:s + target_len] for s in starts]
    energies = np.array([np.sqrt(np.mean(w.astype(np.float64) ** 2)) for w in windows])
    order    = np.argsort(-energies)[:k]
    return [windows[i] for i in order], [starts[i] / sr for i in order]


def _stats(arr):
    return [arr.mean(), arr.std(), arr.min(), arr.max()]


def extract_features(y, sr=32_000):
    target_len = sr * SEGMENT_SEC
    if len(y) > target_len:
        y = y[:target_len]
    else:
        y = np.pad(y, (0, target_len - len(y)), 'constant')

    feat = {}

    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=512))

    mel    = librosa.feature.melspectrogram(S=S**2, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feat['dominant_mel_band'] = float(np.argmax(np.mean(mel_db, axis=1)))
    feat['mel_mean']          = float(mel_db.mean())
    feat['mel_std']           = float(mel_db.std())

    chunk    = sr // 2
    rms_wins = [np.sqrt(np.mean(y[i*chunk:(i+1)*chunk]**2)) for i in range(10)]
    feat['energy_burst_ratio'] = float(np.max(rms_wins) / (np.median(rms_wins) + 1e-6))
    rms = librosa.feature.rms(y=y)[0]
    for k, v in zip(['mean', 'std', 'min', 'max'], _stats(rms)):
        feat[f'rms_{k}'] = float(v)

    mfccs = librosa.feature.mfcc(S=librosa.power_to_db(mel), sr=sr, n_mfcc=20)
    for j in range(20):
        for k, v in zip(['mean', 'std', 'min', 'max'], _stats(mfccs[j])):
            feat[f'mfcc_{j}_{k}'] = float(v)

    delta1 = librosa.feature.delta(mfccs)
    delta2 = librosa.feature.delta(mfccs, order=2)
    for j in range(20):
        feat[f'delta_mfcc_{j}_mean']  = float(delta1[j].mean())
        feat[f'delta_mfcc_{j}_std']   = float(delta1[j].std())
        feat[f'delta2_mfcc_{j}_mean'] = float(delta2[j].mean())
        feat[f'delta2_mfcc_{j}_std']  = float(delta2[j].std())

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    for k, v in zip(['mean', 'std', 'min', 'max'], _stats(centroid)):
        feat[f'centroid_{k}'] = float(v)

    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]
    for k, v in zip(['mean', 'std'], _stats(bandwidth)[:2]):
        feat[f'bandwidth_{k}'] = float(v)

    rolloff_85 = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.85)[0]
    rolloff_95 = librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.95)[0]
    feat['rolloff_85_mean'] = float(rolloff_85.mean())
    feat['rolloff_85_std']  = float(rolloff_85.std())
    feat['rolloff_95_mean'] = float(rolloff_95.mean())

    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)
    for band in range(7):
        feat[f'contrast_{band}_mean'] = float(contrast[band].mean())
        feat[f'contrast_{band}_std']  = float(contrast[band].std())

    flatness = librosa.feature.spectral_flatness(S=S)[0]
    feat['flatness_mean'] = float(flatness.mean())
    feat['flatness_std']  = float(flatness.std())

    zcr = librosa.feature.zero_crossing_rate(y)[0]
    for k, v in zip(['mean', 'std'], _stats(zcr)[:2]):
        feat[f'zcr_{k}'] = float(v)

    return feat


In [3]:

def main():
    print("Loading metadata...")
    train_df = pd.read_csv(os.path.join(DATA_ROOT, 'train.csv'))

    train_df = train_df[train_df['rating'] >= MIN_RATING].reset_index(drop=True)
    print(f"After quality filter: {len(train_df)} rows.")

    if DEBUG_MODE:
        print(f"DEBUG MODE ON: Sampling {N_DEBUG_ROWS} rows.")
        train_df = train_df.sample(N_DEBUG_ROWS, random_state=42).reset_index(drop=True)

    all_rows = []

    print("Extracting original acoustic features (Check logs every 100 rows)...")
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), miniters=100):
        file_path = os.path.join(DATA_ROOT, 'train_audio', row['primary_label'], row['filename'])
        if not os.path.exists(file_path):
            file_path = os.path.join(DATA_ROOT, 'train_audio', row['filename'])

        if not os.path.exists(file_path):
            continue

        y = load_audio_fast(file_path)
        filename = row['filename']

        if np.max(np.abs(y)) == 0:
            continue

        windows, start_secs = top_k_energy_windows(
            y, SAMPLE_RATE, SEGMENT_SEC, k=K_WINDOWS, hop_sec=HOP_SEC
        )

        for window, start_sec in zip(windows, start_secs):
            try:
                feat = extract_features(window, sr=SAMPLE_RATE)
                # Prepend metadata so it can be merged with CV and Perch by (filename, start_sec)
                feat['filename']  = filename
                feat['start_sec'] = float(start_sec)
                all_rows.append(feat)
            except Exception as e:
                print(f"Skipping window for {filename} at {start_sec:.1f}s: {e}")

    df_out = pd.DataFrame(all_rows)

    # Put identifier columns first for readability
    id_cols   = ['filename', 'start_sec']
    feat_cols = [c for c in df_out.columns if c not in id_cols]
    df_out    = df_out[id_cols + feat_cols]

    print(f"\nExtraction complete. Shape: {df_out.shape}")
    df_out.to_parquet(OUT_PARQUET, index=False)
    print(f"Saved to {OUT_PARQUET}")


if __name__ == '__main__':
    main()

Loading metadata...
After quality filter: 22411 rows.
Extracting original acoustic features (Check logs every 100 rows)...


  0%|          | 0/22411 [00:00<?, ?it/s]


Extraction complete. Shape: (60808, 197)
Saved to /kaggle/working/features_v2_with_start_sec.parquet
